In [1]:
import pandas as pd

df = pd.read_parquet(
    "../data/processed/2019-Oct-cleaned.parquet"
)

In [3]:
df.to_parquet(
    "../data/processed/2019-Oct-analysis-ready.parquet",
    index=False
)

In [4]:
import pandas as pd
import numpy as np

df = pd.read_parquet(
    "../data/processed/2019-Oct-cleaned.parquet"
)

df.shape

(42448764, 9)

In [7]:
df["date"] = df["event_time"].dt.floor("D")
df["day"] = df["event_time"].dt.day.astype("uint8")
df["hour"] = df["event_time"].dt.hour.astype("uint8")

df["day_of_week"] = (
    df["event_time"]
    .dt.day_name()
    .astype("category")
)

df["weekend"] = df["event_time"].dt.dayofweek >= 5

df["time_of_day"] = pd.cut(
    df["hour"],
    bins=[-1, 5, 11, 17, 23],
    labels=["Night", "Morning", "Afternoon", "Evening"]
)

In [9]:
df[
    ["event_time", "date", "day", "hour",
     "day_of_week", "weekend", "time_of_day"]
].head()

,event_time,date,day,hour,day_of_week,weekend,time_of_day
0,2019-10-01 00:00:00+00:00,2019-10-01 00:00:00+00:00,1,0,Tuesday,False,Night
1,2019-10-01 00:00:00+00:00,2019-10-01 00:00:00+00:00,1,0,Tuesday,False,Night
2,2019-10-01 00:00:01+00:00,2019-10-01 00:00:00+00:00,1,0,Tuesday,False,Night
3,2019-10-01 00:00:01+00:00,2019-10-01 00:00:00+00:00,1,0,Tuesday,False,Night
4,2019-10-01 00:00:04+00:00,2019-10-01 00:00:00+00:00,1,0,Tuesday,False,Night


In [11]:
category_mapping = pd.DataFrame({
    "category_code": df["category_code"].cat.categories.astype(str)
})

category_parts = (
    category_mapping["category_code"]
    .str.split(".", expand=True)
    .reindex(columns=[0, 1, 2])
)

category_mapping["department"] = category_parts[0]
category_mapping["category"] = category_parts[1]
category_mapping["subcategory"] = category_parts[2]

In [15]:
category_mapping["department"] = (
    category_mapping["department"]
    .fillna("uncategorized")
)

category_mapping["category"] = (
    category_mapping["category"]
    .fillna("unspecified")
)

category_mapping["subcategory"] = (
    category_mapping["subcategory"]
    .fillna("unspecified")
)

In [17]:
category_mapping.head(20)

,category_code,department,category,subcategory
0,accessories.bag,accessories,bag,unspecified
1,accessories.umbrella,accessories,umbrella,unspecified
2,accessories.wallet,accessories,wallet,unspecified
3,apparel.belt,apparel,belt,unspecified
4,apparel.costume,apparel,costume,unspecified
5,apparel.dress,apparel,dress,unspecified
6,apparel.jacket,apparel,jacket,unspecified
7,apparel.jeans,apparel,jeans,unspecified
8,apparel.jumper,apparel,jumper,unspecified
9,apparel.scarf,apparel,scarf,unspecified


In [19]:
department_map = dict(zip(
    category_mapping["category_code"],
    category_mapping["department"]
))

category_map = dict(zip(
    category_mapping["category_code"],
    category_mapping["category"]
))

subcategory_map = dict(zip(
    category_mapping["category_code"],
    category_mapping["subcategory"]
))

In [21]:
df["department"] = (
    df["category_code"]
    .map(department_map)
    .astype("category")
)

df["category"] = (
    df["category_code"]
    .map(category_map)
    .astype("category")
)

df["subcategory"] = (
    df["category_code"]
    .map(subcategory_map)
    .astype("category")
)

In [22]:
df[
    ["category_code", "department", "category", "subcategory"]
].sample(10)

,category_code,department,category,subcategory
14277868,computers.notebook,computers,notebook,unspecified
30309060,electronics.smartphone,electronics,smartphone,unspecified
5585502,uncategorized,uncategorized,unspecified,unspecified
19913486,uncategorized,uncategorized,unspecified,unspecified
39973656,kids.toys,kids,toys,unspecified
91339,uncategorized,uncategorized,unspecified,unspecified
23130521,uncategorized,uncategorized,unspecified,unspecified
15836736,uncategorized,uncategorized,unspecified,unspecified
2716412,electronics.smartphone,electronics,smartphone,unspecified
4305667,furniture.kitchen.chair,furniture,kitchen,chair


In [25]:
df["price"].describe(
    percentiles=[0.25, 0.50, 0.75, 0.90, 0.95, 0.99]
)

count    4.244876e+07
mean     2.903237e+02
std      3.582692e+02
min      0.000000e+00
25%      6.598000e+01
50%      1.629300e+02
75%      3.585700e+02
90%      7.429100e+02
95%      1.010250e+03
99%      1.741340e+03
max      2.574070e+03
Name: price, dtype: float64

In [27]:
positive_prices = df.loc[df["price"] > 0, "price"]

q25 = positive_prices.quantile(0.25)
q50 = positive_prices.quantile(0.50)
q75 = positive_prices.quantile(0.75)

print("25th percentile:", q25)
print("Median:", q50)
print("75th percentile:", q75)

25th percentile: 66.66
Median: 163.97
75th percentile: 358.57


In [29]:
df["price_band"] = pd.cut(
    df["price"],
    bins=[
        -0.001,
        0,
        q25,
        q50,
        q75,
        np.inf
    ],
    labels=[
        "Zero / Unknown",
        "Low",
        "Medium",
        "High",
        "Premium"
    ],
    include_lowest=True
)

In [31]:
df["price_band"].value_counts()

price_band
High              10616051
Medium            10595773
Low               10595165
Premium           10573102
Zero / Unknown       68673
Name: count, dtype: int64

In [33]:
df[["price", "price_band"]].sample(20)

,price,price_band
19014274,1671.34,Premium
4485458,28.06,Low
2949695,46.33,Low
41904269,283.09,High
4111444,77.99,Medium
33085407,68.73,Medium
6019569,15.42,Low
35045195,38.10,Low
7061876,115.96,Medium
19128485,51.22,Low


In [35]:
df["price_band"].isna().sum()

0

In [37]:
df.shape

(42448764, 19)

In [41]:
df.dtypes

event_time       datetime64[ns, UTC]
event_type                  category
product_id                    uint32
category_id                   uint64
category_code               category
brand                       category
price                        float64
user_id                       uint32
user_session                  object
date             datetime64[ns, UTC]
day                            uint8
hour                           uint8
day_of_week                 category
weekend                         bool
time_of_day                 category
department                  category
category                    category
subcategory                 category
price_band                  category
dtype: object

In [43]:
df.isna().sum()

event_time       0
event_type       0
product_id       0
category_id      0
category_code    0
brand            0
price            0
user_id          0
user_session     2
date             0
day              0
hour             0
day_of_week      0
weekend          0
time_of_day      0
department       0
category         0
subcategory      0
price_band       0
dtype: int64

In [45]:
print("Rows:", len(df))
print("Unique users:", df["user_id"].nunique())
print("Unique products:", df["product_id"].nunique())
print("Purchases:", (df["event_type"] == "purchase").sum())

Rows: 42448764
Unique users: 3022290
Unique products: 166794
Purchases: 742849


In [47]:
memory_after_transform = (
    df.memory_usage(deep=True).sum() / (1024 ** 3)
)

print(f"Memory usage: {memory_after_transform:.2f} GB")

Memory usage: 5.81 GB


In [49]:
df.to_parquet(
    "../data/processed/2019-Oct-analysis-ready.parquet",
    index=False
)

In [50]:
df.head()

,event_time,event_type,product_id,category_id,category_code,brand,price,user_id,user_session,date,day,hour,day_of_week,weekend,time_of_day,department,category,subcategory,price_band
0,2019-10-01 00:00:00+00:00,view,44600062,2103807459595387724,uncategorized,shiseido,35.79,541312140,72d76fde-8bb3-4e00-8c23-a032dfed738c,2019-10-01 00:00:00+00:00,1,0,Tuesday,False,Night,uncategorized,unspecified,unspecified,Low
1,2019-10-01 00:00:00+00:00,view,3900821,2053013552326770905,appliances.environment.water_heater,aqua,33.20,554748717,9333dfbd-b87a-4708-9857-6336556b0fcc,2019-10-01 00:00:00+00:00,1,0,Tuesday,False,Night,appliances,environment,water_heater,Low
2,2019-10-01 00:00:01+00:00,view,17200506,2053013559792632471,furniture.living_room.sofa,unknown,543.10,519107250,566511c2-e2e3-422b-b695-cf8e6e792ca8,2019-10-01 00:00:00+00:00,1,0,Tuesday,False,Night,furniture,living_room,sofa,Premium
3,2019-10-01 00:00:01+00:00,view,1307067,2053013558920217191,computers.notebook,lenovo,251.74,550050854,7c90fc70-0e80-4590-96f3-13c02c18c713,2019-10-01 00:00:00+00:00,1,0,Tuesday,False,Night,computers,notebook,unspecified,High
4,2019-10-01 00:00:04+00:00,view,1004237,2053013555631882655,electronics.smartphone,apple,1081.98,535871217,c6bd7419-2748-4c56-95b4-8cec9ff8b80d,2019-10-01 00:00:00+00:00,1,0,Tuesday,False,Night,electronics,smartphone,unspecified,Premium


# STEP 4 — Data Transformation & Feature Engineering

## Objective

The purpose of Day 4 was to transform the cleaned October 2019 e-commerce dataset into an analysis-ready format.

No rows were removed during transformation.

## Dataset Size

Before transformation:

- **42,448,764 rows**
- **9 columns**

After transformation:

- **42,448,764 rows**
- **19 columns**

This confirms that transformation added analytical features without removing any original behavioural records.

## Time Features

The following fields were created from `event_time`:

- `date`
- `day`
- `hour`
- `day_of_week`
- `weekend`
- `time_of_day`

These fields will support daily, hourly, weekday/weekend, and time-of-day behavioural analysis.

## Product Category Transformation

The hierarchical `category_code` field was separated into:

- `department`
- `category`
- `subcategory`

For example:

`apparel.shoes.keds`

was transformed into:

- Department: `apparel`
- Category: `shoes`
- Subcategory: `keds`

Where a hierarchy level was unavailable, an appropriate `unspecified` value was retained rather than inventing a category.

## Price Band Feature

A `price_band` field was created using the distribution of positive product prices.

The bands are:

- Zero / Unknown
- Low
- Medium
- High
- Premium

The thresholds were based on dataset quartiles rather than arbitrary price values.

This allows customer behaviour to be compared across relative price ranges.

## Final Data Types

The transformed dataset uses memory-conscious datatypes including:

- datetime for event timestamps and dates
- categorical types for low-cardinality text fields
- unsigned integer types for identifiers and time fields
- Boolean for the weekend flag
- float64 for price

## Missing Values After Transformation

All newly created features contain **0 missing values**.

The only remaining missing information in the complete dataset is:

- `user_session`: **2 records**

These two values were intentionally retained because the correct session identifiers could not be reconstructed reliably.

## Validation

The transformation process successfully preserved:

- all original rows
- all users
- all products
- all behavioural events

No records were lost as a result of feature engineering.
